# REWIND M2 Train300 / 400 updates
검증된 M2_scaling300_gpu_input.zip을 업로드하세요. 런타임은 T4 GPU입니다. 기존 Train18/100 재학습 금지. 신규 Train300 한 번만 실행합니다.
선택·8개 기준·최종400 checkpoint는 학습 전에 고정했습니다. 원본 mocap 독립성은 NOT VERIFIED.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import files
ROOT = Path('/content/rewind_scaling300')
assert not ROOT.exists(), '이미 준비된 실행입니다. 새 학습을 반복하지 마세요.'
uploaded = files.upload()
name = 'M2_scaling300_gpu_input.zip'
assert name in uploaded, name
blob = uploaded[name]
assert len(blob) == 7614593
assert hashlib.sha256(blob).hexdigest() == '2a4f1211befe505e9a02ec31e9b32f930e2b5a78fd903c1aa7714b0be3a3f5a2'
ROOT.mkdir()
bundle = ROOT / name
bundle.write_bytes(blob)
with zipfile.ZipFile(bundle) as z:
    assert z.testzip() is None
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(ROOT/'private')
for n,h in json.loads((ROOT/'private/bundle_checksums.json').read_text()).items():
    assert hashlib.sha256((ROOT/'private'/n).read_bytes()).hexdigest() == h
subprocess.run(['git','clone','--branch','experiment/m2-dataset-scaling-300-20261117','--single-branch','https://github.com/darkkyoung/REWIND-like-prototype.git',str(ROOT/'repo')],check=True)
SOURCE_COMMIT = 'ea762262507b79999b69b37ff9f421e9bc0be94e'
subprocess.run(['git','checkout','--detach',SOURCE_COMMIT],cwd=ROOT/'repo',check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT/'repo',text=True).strip()==SOURCE_COMMIT
print('INPUT VERIFIED', len(json.loads((ROOT/'private/bundle_checksums.json').read_text())))



In [ ]:
import torch, unittest
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(), '실제 T4가 필요합니다. CPU로 학습하지 마세요.'
print('GPU',torch.cuda.get_device_name(), 'torch',torch.__version__, 'CUDA',torch.version.cuda)
os.chdir(ROOT/'repo');sys.path.insert(0,str(ROOT/'repo'))
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['M2_SCALING300_INPUT']=str(ROOT/'private/input')
os.environ['M2_SCALING300_RAW']=str(ROOT/'private/raw_pose')
test_code = """import torch,unittest
torch.set_num_threads(1)
loader=unittest.TestLoader()
suite=unittest.TestSuite([loader.discover('tests', pattern='test_m2.py'),loader.loadTestsFromName('experiments.final_m2.dataset_scaling_300.test_scaling')])
r=unittest.TextTestRunner(verbosity=2).run(suite)
raise SystemExit(not r.wasSuccessful())
"""
r=subprocess.run([sys.executable,'-c',test_code],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(ROOT/'colab_regression_tests.log').write_text(r.stdout)
print(r.stdout);assert r.returncode==0



In [ ]:
# Train300 한 군의 400 updates. 결과 디렉터리 존재 시 재학습하지 않는다.
out=ROOT/'results'
assert not out.exists(), '기존 실행이 있습니다. 재학습하지 마세요.'
cmd=[sys.executable,'-u','experiments/final_m2/dataset_scaling_300/run_experiment.py','--input-dir',str(ROOT/'private/input'),'--baseline-dir',str(ROOT/'private/baseline'),'--output',str(out)]
with (ROOT/'console.log').open('w') as log:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='');log.write(line);log.flush()
    returncode=proc.wait()
print('PROCESS_RETURN_CODE',returncode)
# 성공 여부와 무관하게 실제 생성된 파일을 회수한다. 로그를 만들거나 보충하지 않는다.
import shutil
if out.exists():
    shutil.copy2(ROOT/'console.log',out/'console.log')
    shutil.copy2(ROOT/'colab_regression_tests.log',out/'colab_regression_tests.log')
    (out/'checksums.json').write_text(json.dumps({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file() and p.name!='checksums.json'},indent=2))
archive=ROOT/'REWIND_M2_Scaling300_T4_raw.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,6) as z:
    for p in sorted(out.iterdir()):
        if p.is_file():z.write(p,p.name)
with zipfile.ZipFile(archive) as z:assert z.testzip() is None
receipt={'file':archive.name,'bytes':archive.stat().st_size,'sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'returncode':returncode,'source_commit':SOURCE_COMMIT}
(ROOT/'raw_archive_receipt.json').write_text(json.dumps(receipt,indent=2));print(json.dumps(receipt))
from IPython.display import display,FileLink
display(FileLink(str(archive),result_html_prefix='Download verified scaling300 raw T4 results: '))
files.download(str(archive))
assert returncode==0, '실패 기록을 회수하세요. 자동으로 재학습하지 마세요.'

